 # 00_pull notebook

### Loads in raw survey data for both Brazil and South Africa, and produces a stacked dataset with col names organized properly.
#### **South Africa**
Notebook takes Quartely Labour Force Survey from South Africa for 2008, 2010, 2012. The government of South Africa releases results quarterly, so to standarize, I take from Q4 each year. Those survey data are then stacked into one dataframe.

#### **Brazil**
Notebook takes PNAD Continua (the national household survey) data from Brazil for 2012, 2014, 2016. As with South Africa, the government of Brazil releases results quarterly, so to standarize, I take from Q4 each year. Those survey data are then stacked into one dataframe. PNAD Continua organizes its data on a platform called SIDRA, where you download only specific modules matching the information you want. I downloaded 4, all of which are explained in the README. SIDRA downloads also carry a bunch of header rows that are unimportant and mess up the standarizing, so I drop those here as well. I also translate the col names out of Portugese

#### **Inputs**
data/southafrica_2008.dta, data/southafrica_2010.dta, data/southafrica_2012.dta
data/brazil_5439.xlsx, data/brazil_4093.xlsx, data/brazil_5947.xlsx, data/brazil_6371.xlsx

#### **Outputs**
data/southafrica_all.dta
data/brazil_lstatus.csv, data/brazil_socialsecurity.csv, data/brazil_income.csv, data/brazil_hours.csv

In [55]:
import pandas as pd
# create data name for the path variable, so others can execute code (per instructions on final project)
data = "../data/" # easier to use going forward
# create list for south africa appropriate years (this makes the stacking way easier)
southafrica_years = [2008, 2010, 2012]

# write function for the SIDRA tables
# done because i am cleaning 4 tables that are formatted the same way the exact same, make it faster with function
def load_sidra(filename, col_names):
    brazil_df = pd.read_excel(data + filename, header=None, skiprows=5, names=col_names) # skip the 5 header rows i dont want
    brazil_df = brazil_df.iloc[:20]   # keep the 20 metro rows, drop SIDRA source note that is the last row
    return brazil_df

### South Africa

In [60]:
# load data on South Africa labor stats
# first, make list of all the cols i care about from dataset
southafrica_cols = ["Metro_code", "Sector2", "Hrswrk", "Weight","Province", "Q13GENDER", "Q14AGE", "Q15POPULATION", "Q17EDUCATION", "Indus"]

# for loop to iterate through each year in list from earlier
southafrica_list = []
for year in southafrica_years:
    # 2012 spells this column Status_exp, 2008 and 2010 use Status_Exp
    status_col = "Status_exp" if year == 2012 else "Status_Exp"
    df = pd.read_stata(data + f"southafrica_{year}.dta", columns=southafrica_cols + [status_col])
    df.columns = df.columns.str.lower()   # make it all lowercase (it flips back and forth across years)
    df["hrswrk"] = pd.to_numeric(df["hrswrk"], errors="coerce")  # convert to numeric so stata is happy, coerce so it doesnt stop, NaN instead
    df["year"] = year # create year variable so it can be stacked
    southafrica_list.append(df) # add df to list to be concated in next step

## stack the three years into one dataframe with added year column
southafrica_df = pd.concat(southafrica_list, ignore_index=True)
print(southafrica_df.shape)
print(southafrica_df.groupby("year").size())
print(southafrica_df["metro_code"].unique())
print(southafrica_df["sector2"].unique()) # informal employment variable

(261531, 12)
year
2008    93062
2010    83357
2012    85112
dtype: int64
['Non-Metro' ' Cape Town' ' Nelson Mandela Metro' ' eThekweni' 'Tshwane'
 'eKhurhuleni' ' Johannesburg' 'Non_Metro' 'Cape Town'
 'Nelson Mandela Metro' 'eThekwini' 'eKurhuleni' 'Johannesburg'
 'eThekweni']
['Formal sector (Including agriculture)', 'Not applicable', 'Private households', 'Informal sector (Including agriculture)']
Categories (4, object): ['Not applicable' < 'Formal sector (Including agriculture)' < 'Informal sector (Including agriculture)' < 'Private households']


### Brazil

In [61]:
# run function written earlier
# function takes in the file name, and then list of the col names
# 5947-- contribution to social security (formality measure for labor/employment)
# emp=everyone working, contrib=workers contributing to social security (formal work), noncontrib=workers NOT contributing (informal work)
contrib_cols = ["metro","emp_2012","contrib_2012", "noncontrib_2012", "emp_2014", "contrib_2014", "noncontrib_2014",
     "emp_2016", "contrib_2016", "noncontrib_2016"]
brazil_contrib = load_sidra("5947_brazil.xlsx", contrib_cols)

# 5439-- average real monthly income
# inc_total=ave income across everyone in given year; inc_employee=ave income across employees, 
# inc_employer=ave income acorss employers, inc_own=ave income across self employed 
income_cols = ["metro","inc_total_2012", "inc_employee_2012", "inc_employer_2012", "inc_own_2012",
     "inc_total_2014", "inc_employee_2014", "inc_employer_2014", "inc_own_2014",
     "inc_total_2016", "inc_employee_2016", "inc_employer_2016", "inc_own_2016"]
brazil_income = load_sidra("5439_brazil.xlsx",income_cols)

# 6371-- average hours usually worked per week for primary job
hours_cols = ["metro", "hours_2012", "hours_2014", "hours_2016"]
brazil_hours = load_sidra("6371_brazil.xlsx", hours_cols)

# 4093-- labor force status (unemployment counts)
# total=unemployed+employed; emp=employed; unemp=unemployed; na=not in labor force
status_cols = ["level", "code", "metro","total_2012", "emp_2012", "unemp_2012", "na_2012",
     "total_2014", "emp_2014", "unemp_2014", "na_2014","total_2016", "emp_2016", "unemp_2016", "na_2016"]
brazil_status = load_sidra("4093_brazil.xlsx", status_cols)

# check all 4 loaded 20 metro rows
# should be (20, 10), (20, 13), (20, 4), (20, 15)
for brazil_df in [brazil_contrib, brazil_income, brazil_hours, brazil_status]:
    print(brazil_df.shape) 

(20, 10)
(20, 13)
(20, 4)
(20, 15)


### Save output for following notebooks

In [62]:
southafrica_df.to_stata(data + "southafrica_all.dta", write_index=False)
brazil_status.to_csv(data + "brazil_lstatus.csv", index=False)
brazil_contrib.to_csv(data + "brazil_socialsecurity.csv", index=False)
brazil_income.to_csv(data + "brazil_income.csv", index=False)
brazil_hours.to_csv(data + "brazil_hours.csv", index=False)
print("Saved files")

Saved files
